In [1]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import json
import pickle

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.optimizers import SGD   
import random

words=[]
classes = []
documents = []
ignore_words = ['?', '!']
data_file = open('intents.json').read()
intents = json.loads(data_file)


for intent in intents['intents']:
    for pattern in intent['patterns']:

        # take each word and tokenize it
        w = nltk.word_tokenize(pattern)
        words.extend(w)
        # adding documents
        documents.append((w, intent['tag']))

        # adding classes to our class list
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))

classes = sorted(list(set(classes)))

print (len(documents), "documents")

print (len(classes), "classes", classes)

print (len(words), "unique lemmatized words", words)


pickle.dump(words,open('wordsJ.pkl','wb'))
pickle.dump(classes,open('classesJ.pkl','wb'))

# initializing training data
training = []
output_empty = [0] * len(classes)
for doc in documents:
    # initializing bag of words
    bag = []
    # list of tokenized words for the pattern
    pattern_words = doc[0]
    # lemmatize each word - create base word, in attempt to represent related words
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    # create our bag of words array with 1, if word match found in current pattern
    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)

    # output is a '0' for each tag and '1' for current tag (for each pattern)
    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1

    training.append([bag, output_row])
# shuffle our features and turn into np.array
random.shuffle(training)
#training = np.array(training)
# create train and test lists. X - patterns, Y - intents
train_x = [item[0] for item in training]
train_y = [item[1] for item in training]
print("Training data created")


# Create model - 3 layers. First layer 128 neurons, second layer 64 neurons and 3rd output layer contains number of neurons
# equal to number of intents to predict output intent with softmax
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))

# Compile model. Stochastic gradient descent with Nesterov accelerated gradient gives good results for this model
sgd = SGD(lr=0.01, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

#fitting and saving the model
hist = model.fit(np.array(train_x), np.array(train_y), epochs=300, batch_size=5, verbose=1)
model.save('chatbot_modelJ.h5', hist)

print("model created")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Madhan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Madhan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


209 documents
67 classes ['career_advancement_strategies', 'career_advice_for_entrepreneurs', 'career_advice_for_freelancers', 'career_advice_from_mentors', 'career_advice_sources', 'career_advisor_introduction', 'career_burnout_prevention', 'career_change_advice', 'career_change_advice_for_veterans', 'career_change_at_midlife', 'career_change_decision_making', 'career_gap_handling', 'career_goal', 'career_growth', 'career_growth_evaluation', 'career_growth_in_nontraditional_fields', 'career_growth_strategies', 'career_mentorship', 'career_path_exploration', 'career_plateau', 'career_portfolio_building', 'career_progress_evaluation', 'career_reentry_after_caregiving', 'career_skills_development', 'career_transition', 'career_transition_advice', 'career_transition_after_redundancy', 'career_transition_strategies', 'career_transition_to_leadership_role', 'goodbye', 'handling_workplace_conflicts', 'improving_professional_network', 'interview_preparation', 'job_interview_preparation', 'job

D:\anaconda\envs\env\lib\site-packages\keras\optimizers\optimizer_v2\gradient_descent.py:108: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


42/42 [==============================] - 1s 2ms/step - loss: 4.2309 - accuracy: 0.0239
Epoch 2/300
42/42 [==============================] - 0s 2ms/step - loss: 4.2094 - accuracy: 0.0144
Epoch 3/300
42/42 [==============================] - 0s 2ms/step - loss: 4.1552 - accuracy: 0.0383
Epoch 4/300
42/42 [==============================] - 0s 2ms/step - loss: 4.1253 - accuracy: 0.0335
Epoch 5/300
42/42 [==============================] - 0s 2ms/step - loss: 4.0767 - accuracy: 0.0670
Epoch 6/300
42/42 [==============================] - 0s 2ms/step - loss: 4.0645 - accuracy: 0.0335
Epoch 7/300
42/42 [==============================] - 0s 2ms/step - loss: 3.9710 - accuracy: 0.0574
Epoch 8/300
42/42 [==============================] - 0s 2ms/step - loss: 3.8569 - accuracy: 0.1005
Epoch 9/300
42/42 [==============================] - 0s 2ms/step - loss: 3.8445 - accuracy: 0.0909
Epoch 10/300
42/42 [==============================] - 0s 2ms/step - loss: 3.6898 - accuracy: 0.1627
Epoch 11/300
42/42 [=